# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR² dataset) Exploration with `mlcroissant`
This notebook demonstrates step-by-step loading, inspection, processing, and visualization of the FAIR² dataset using the `mlcroissant` library, referencing all fields strictly by their Croissant `@id` values as specified in the schema.

### Dataset Source
The dataset schema is provided via a Croissant JSON-LD URL for machine-actionable discovery and loading.

In [ ]:
# Ensure `mlcroissant` is installed in this environment
!pip install mlcroissant --quiet

## 1. Data Loading
Load dataset metadata and discover available data using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset (schema + metadata)
dataset = mlc.Dataset(croissant_url)
# Extract metadata object
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")
if hasattr(metadata, 'identifier'):
    print(f"DOI: {metadata.identifier}")
if hasattr(metadata, 'keywords'):
    print(f"Keywords: {', '.join(metadata.keywords)}")
if hasattr(metadata, 'datePublished'):
    print(f"Date Published: {metadata.datePublished}")

## 2. Data Overview
Discover available record sets, their Croissant `@id`s, and the fields/classes (`cr:field`) within each recordset. All entities are referenced by their `@id` for reproducibility and clarity, as recommended in best practices.

We'll print all available record sets and their contained fields (columns), and show the type/schema for each field where possible.

In [ ]:
# List available record sets with their @id and fields' @id
recordset_info = []
for recset in dataset.record_sets:
    print(f"\nRecord set: {recset['@id']}")
    print(f"  Name: {recset.get('name','')} | Description: {recset.get('description','')}")
    field_list = []
    # Fields are typically under cr:field (or sometimes just 'field')
    for field in recset.get('field', []) + recset.get('cr:field', []):
        field_id = field.get('@id') if isinstance(field, dict) else str(field)
        print(f"    Field @id: {field_id}")
        if isinstance(field, dict):
            print(f"      Name: {field.get('name','')} | dataType: {field.get('dataType','')}")
        field_list.append(field_id)
    recordset_info.append({'@id': recset['@id'], 'fields': field_list})

## 3. Data Extraction
Extract data from specific record sets, referencing both the record set and fields by their `@id` as above. We'll demonstrate loading the main table (pick the primary record set for the cohort and clinical variables). All data is loaded into DataFrames for further analysis.

In [ ]:
# Prepare a list of all record set @ids (from the previous cell, recordset_info)
recordset_ids = [recset['@id'] for recset in recordset_info]
dataframes = {}

for recset_id in recordset_ids:
    recs = list(dataset.records(record_set=recset_id))
    if recs:  # Only build DataFrame if there are records
        df = pd.DataFrame(recs)
        dataframes[recset_id] = df
        print(f"Loaded record set: {recset_id}, shape: {df.shape}")

# Choose a main record set for further analysis (example: the first with at least one row)
main_recset_id = None
for rid, df in dataframes.items():
    if len(df) > 0:
        main_recset_id = rid
        break

if main_recset_id:
    print(f"\nMain record set chosen for EDA: {main_recset_id}")
    print("Available field @ids:", dataframes[main_recset_id].columns.tolist())
    dataframes[main_recset_id].head()
else:
    print("No record sets with data found.")

## 4. Exploratory Data Analysis (EDA)
We apply example processing using `@id` references. We'll pick a numeric field (e.g., `age` or `interval_years`) and a categorical field (e.g., `sex` or `msi_status`, depending on what's in the schema) _by their real `@id`_, filter on a value, normalize, and group by category.

**Note**: Replace the variables below with the specific `@id`s that match the numeric and group/categorical fields in your chosen record set.

In [ ]:
# Select field @ids present in main_recset_id's DataFrame for demonstration
# Please update these @ids according to the real schema/fields printed above.

# Let's try to infer a numeric and group field from the columns, fallback to guessing common names
numeric_field_id = None
group_field_id = None

for col in dataframes[main_recset_id].columns:
    if ('age' in col.lower() or 'interval' in col.lower() or 'years' in col.lower()) and numeric_field_id is None:
        numeric_field_id = col
    if ('sex' in col.lower() or 'msi' in col.lower() or 'group' in col.lower() or 'location' in col.lower()) and group_field_id is None:
        group_field_id = col

if numeric_field_id is None:
    # Fall back to first column (may not be numeric)
    numeric_field_id = dataframes[main_recset_id].columns[0]
if group_field_id is None and len(dataframes[main_recset_id].columns) > 1:
    group_field_id = dataframes[main_recset_id].columns[1]

print(f"Using field for numeric analysis: {numeric_field_id}")
print(f"Using field for grouping: {group_field_id}")

# Attempt numeric operations, handling conversion
df = dataframes[main_recset_id].copy()
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalization
if filtered_df[numeric_field_id].notnull().any():
    norm_field = f"{numeric_field_id}_normalized"
    filtered_df[norm_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_field]].head())

# Group by group_field_id
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nAverage {numeric_field_id} grouped by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Let's visualize a distribution of a numeric field (e.g., age or interval) and show group statistics by another field. All references are `@id` from the Croissant schema.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

if group_field_id in df.columns and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
- We have successfully loaded the FAIR² dataset defined by a Croissant schema, explored its record sets and fields strictly using their `@id` references, and performed basic exploratory analysis and visualizations.
- This workflow provides a reproducible, machine-actionable process for FAIR dataset access and secondary analysis.
- For further analysis, consult the Croissant schema (`@id`s) to work precisely with additional fields, reference documentation, or transform output as needed.